# DEFCON — one-match demo

End-to-end, top to bottom: **raw Metrica data → component models → EPV → per-player defensive credit.**

**Prerequisites** (run once from the repo root):
```bash
pip install -e ".[dev,gnn,boost,viz]"
python scripts/download_wyscout.py && python scripts/download_metrica.py
python scripts/train_uxg.py && python scripts/train_pass_success.py && python scripts/train_components.py
```
This notebook then loads the trained checkpoints and runs the valuation pipeline on Game 1.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline
from defcon import load_config
cfg = load_config()
GAME_DIR = cfg.path("tracking_dir") / "metrica"
print("pitch:", cfg.pitch.length, "x", cfg.pitch.width, "m  |  frame rate:", cfg.tracking.frame_rate, "Hz")

## 1 · Load and process a match
Parse tracking + events, sync, add kinematics, label, cache.

In [ ]:
from defcon.data.pipeline import process_metrica_match
actions, tracking = process_metrica_match(GAME_DIR, game_id=1, cfg=cfg)
print(f"tracking rows: {len(tracking):,}   on-ball actions: {len(actions)}")
passes = actions[actions.type == "pass"]
print(f"passes: {len(passes)}  (completion {passes.outcome.eq('success').mean():.2f})   "
      f"shots: {(actions.type=='shot').sum()}")
actions[["action_id","type","outcome","team","player","inferred_receiver","scores_next"]].head()

## 2 · A tracking frame
Unified schema → a plausible 11 v 11 shape, ball at a player's feet.

In [ ]:
from defcon.viz.pitch import plot_frame
plot_frame(tracking, frame=52000, cfg=cfg);

## 3 · UxG — unblocked expected goals
Standalone logistic-regression xG model (trained on Wyscout).

In [ ]:
from defcon.models.uxg import UxGModel
uxg = UxGModel.load(cfg.path("checkpoints") / "uxg.joblib", cfg)
for label, x, y in [("penalty spot", cfg.pitch.length-11, cfg.pitch.width/2),
                    ("6-yard box",  cfg.pitch.length-6,  cfg.pitch.width/2),
                    ("25 m out",    cfg.pitch.length-25, cfg.pitch.width/2)]:
    print(f"{label:14s}  UxG = {uxg.score_location(x, y):.3f}")

## 4 · Expected Possession Value
Assemble the models into a per-option table (Fig-2 style) and a scalar state EPV.

In [ ]:
from defcon.epv.epv import EPVEngine
from defcon.data.metrica import infer_playing_direction, identify_goalkeepers
from defcon.features.state import graph_state_from_action

epv = EPVEngine.from_checkpoints(cfg)
directions, gks = infer_playing_direction(tracking), identify_goalkeepers(tracking)
row = passes.reset_index(drop=True).iloc[300]
state = graph_state_from_action(tracking, row, directions, gks, cfg)
table = epv.option_table(state).sort_values("p_select", ascending=False)
print(f"state EPV = {epv.state_epv(state):.3f}")
table.head(6).round(3)

In [ ]:
# EPV rises toward the opponent goal (see docs/img/epv_surface.png for the full surface):
from IPython.display import Image
Image("docs/img/epv_surface.png")

## 5 · Defensive credit
Team value `D = EPV_k − EPV_{k+1}`, routed to a scenario rule, split onto defenders, tagged **Intercept / Disturb / Deter / Concede**, normalized per 90.

In [ ]:
from defcon.credit.engine import CreditEngine
from defcon.credit.aggregate import aggregate_per90, credits_to_items, minutes_played

engine = CreditEngine(epv, cfg)
credits = engine.process_match(actions, tracking)
print("scenario mix:", credits.drop_duplicates('action_id')['case'].value_counts().to_dict())

items = credits_to_items(credits)
mins = minutes_played(tracking, cfg)
pt = actions.dropna(subset=['player']).groupby('player')['team'].agg(lambda s: s.mode().iloc[0]).to_dict()
table90 = aggregate_per90(items, mins, pt, min_minutes=20)
table90[["player","team","intercept_p90","disturb_p90","deter_p90","concede_p90","net_p90"]].round(2).head(10)

In [ ]:
# Figure-6: per-player stacked credit
from defcon.viz.credit import plot_credit_breakdown
from IPython.display import Image
plot_credit_breakdown(table90, "docs/img/_demo_fig6.png", cfg, top_n=12,
                      title="Defensive credit — demo (Game 1)")
Image("docs/img/_demo_fig6.png")

## Caveats (read me)

This runs on **2 anonymized public matches**, so it is a *method* reproduction, not a *results* one:

- Rare-event models (**c2** conceding, **d1** responsibility) are data-starved.
- Absolute credit magnitudes inherit the c1 calibration offset (shape is correct; see `docs/SCOPE.md`).
- The paper's headline **market-value validation cannot be run** — Metrica players are anonymized.

The credit *rules* themselves are exact: the paper's Fig-3 and Fig-5 worked examples are reproduced to
the digit in `tests/test_credit_rules.py`.